# Household Energy Consumption Analysis and Prediction

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.cluster import KMeans

### Loading Data

In [ ]:
df = pd.read_csv('household_power_consumption.txt', sep=';', na_values=['?'], low_memory=False)

In [ ]:
df.head()

In [ ]:
df.shape

### Preprocessing

In [ ]:
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')

In [ ]:
df.drop(columns=['Date', 'Time'], inplace=True)

In [ ]:
df.set_index('Datetime', inplace=True)

In [ ]:
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [ ]:
df.isna().sum()

In [ ]:
df.ffill(inplace=True)

### Hourly Resampling and Feature Extraction

In [ ]:
df_hourly = df.resample('h').mean()

In [ ]:
df_hourly.ffill(inplace=True)

In [ ]:
df_hourly['Hour'] = df_hourly.index.hour
df_hourly['DayOfWeek'] = df_hourly.index.dayofweek
df_hourly['Month'] = df_hourly.index.month
df_hourly['Is_Weekend'] = (df_hourly['DayOfWeek'] >= 5).astype(int)

In [ ]:
df_hourly.head()

In [ ]:
df_hourly.shape

### Correlation Analysis

In [ ]:
p_corr = df_hourly.corr(method='pearson')['Global_active_power']
s_corr = df_hourly.corr(method='spearman')['Global_active_power']
k_corr = df_hourly.corr(method='kendall')['Global_active_power']

In [ ]:
corr_df = pd.DataFrame({'Pearson': p_corr, 'Spearman': s_corr, 'Kendall': k_corr}).drop('Global_active_power')
corr_df

In [ ]:
corr_df.plot(kind='bar', figsize=(10, 4))
plt.title('Correlation with Global Active Power')
plt.ylabel('Score')
plt.grid(True, alpha=0.3)
plt.show()

### Exploratory Plots

In [ ]:
hourly_power = df_hourly.groupby('Hour')['Global_active_power'].mean()

In [ ]:
plt.plot(hourly_power.index, hourly_power.values, marker='o')
plt.title('Average Power by Hour')
plt.xlabel('Hour')
plt.ylabel('kW')
plt.grid(True)
plt.show()

In [ ]:
sub_hourly = df_hourly.groupby('Hour')[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

In [ ]:
plt.plot(sub_hourly['Sub_metering_1'], label='Kitchen')
plt.plot(sub_hourly['Sub_metering_2'], label='Laundry')
plt.plot(sub_hourly['Sub_metering_3'], label='AC/Heater')
plt.title('Sub-metering Usage by Hour')
plt.xlabel('Hour')
plt.ylabel('Watt-hours')
plt.legend()
plt.grid(True)
plt.show()

### Data Splitting and Scaling

In [ ]:
features = ['Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3', 'Hour', 'DayOfWeek', 'Month', 'Is_Weekend']
X = df_hourly[features]
y = df_hourly['Global_active_power']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### 1. Linear Regression

In [ ]:
lr = LinearRegression()
lr.fit(X_train_s, y_train)
lr_pred = lr.predict(X_test_s)

In [ ]:
lr_r2 = r2_score(y_test, lr_pred)
lr_mae = mean_absolute_error(y_test, lr_pred)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_pred))

print('Linear Regression R2:', lr_r2)
print('Linear Regression MAE:', lr_mae)
print('Linear Regression RMSE:', lr_rmse)

### 2. Ridge Regression

In [ ]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)
ridge_pred = ridge.predict(X_test_s)

In [ ]:
ridge_r2 = r2_score(y_test, ridge_pred)
ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))

print('Ridge R2:', ridge_r2)
print('Ridge MAE:', ridge_mae)
print('Ridge RMSE:', ridge_rmse)

### 3. Lasso Regression

In [ ]:
lasso = Lasso(alpha=0.001)
lasso.fit(X_train_s, y_train)
lasso_pred = lasso.predict(X_test_s)

In [ ]:
lasso_r2 = r2_score(y_test, lasso_pred)
lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))

print('Lasso R2:', lasso_r2)
print('Lasso MAE:', lasso_mae)
print('Lasso RMSE:', lasso_rmse)

### 4. Decision Tree Regressor

In [ ]:
dt = DecisionTreeRegressor(max_depth=8, random_state=42)
dt.fit(X_train_s, y_train)
dt_pred = dt.predict(X_test_s)

In [ ]:
dt_r2 = r2_score(y_test, dt_pred)
dt_mae = mean_absolute_error(y_test, dt_pred)
dt_rmse = np.sqrt(mean_squared_error(y_test, dt_pred))

print('Decision Tree R2:', dt_r2)
print('Decision Tree MAE:', dt_mae)
print('Decision Tree RMSE:', dt_rmse)

### 5. Random Forest Regressor

In [ ]:
rf = RandomForestRegressor(n_estimators=40, max_depth=8, random_state=42)
rf.fit(X_train_s, y_train)
rf_pred = rf.predict(X_test_s)

In [ ]:
rf_r2 = r2_score(y_test, rf_pred)
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))

print('Random Forest R2:', rf_r2)
print('Random Forest MAE:', rf_mae)
print('Random Forest RMSE:', rf_rmse)

### Model Comparison

In [ ]:
results_data = {
    'Model': ['Linear Regression', 'Ridge', 'Lasso', 'Decision Tree', 'Random Forest'],
    'MAE': [lr_mae, ridge_mae, lasso_mae, dt_mae, rf_mae],
    'RMSE': [lr_rmse, ridge_rmse, lasso_rmse, dt_rmse, rf_rmse],
    'R2': [lr_r2, ridge_r2, lasso_r2, dt_r2, rf_r2]
}
res_df = pd.DataFrame(results_data)
res_df

In [ ]:
plt.bar(res_df['Model'], res_df['R2'])
plt.title('Model R2 Comparison')
plt.ylim(0.9, 1.0)
plt.xticks(rotation=20)
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:100], label='Actual', color='black')
plt.plot(lr_pred[:100], label='Linear Regression', color='red', linestyle='--')
plt.title('Actual vs Predicted Power (First 100 Hours)')
plt.legend()
plt.show()

### K-Means Clustering

In [ ]:
cluster_cols = ['Global_active_power', 'Global_intensity', 'Voltage', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
X_c = df_hourly[cluster_cols]

In [ ]:
c_scaler = StandardScaler()
X_c_s = c_scaler.fit_transform(X_c)

In [ ]:
wcss = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_c_s)
    wcss.append(km.inertia_)

In [ ]:
plt.plot(range(2, 7), wcss, marker='o')
plt.title('Elbow Method')
plt.xlabel('K')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_hourly['Cluster'] = km_final.fit_predict(X_c_s)

In [ ]:
df_hourly.groupby('Cluster')[cluster_cols].mean()

In [ ]:
sns.countplot(data=df_hourly, x='Hour', hue='Cluster')
plt.title('Cluster Distribution by Hour')
plt.show()